In [3]:
# Import required libraries for file system operations, image loading, and PyTorch inference
import os
import cv2
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

### Dataset Overview

The test dataset contains unseen CAPTCHA images stored in `test_images/`. The objective is to recognize the character sequence in each image using the custom model trained from scratch.

In [4]:
# Specify paths for test images directory and the trained model weights checkpoint
TEST_FOLDER = "/kaggle/input/datasets/hardikgautam24119014/datasetnew/cig_ps/test_images"

MODEL_PATH = "/kaggle/input/models/hardikgautam24119014/final-model/pytorch/default/1/final_resnet18_captcha.pth"

### Data Exploration

Exploration of the test set is limited to loading the image file names and confirming the test directory structure. Detailed data exploration and visualization are performed in `01_Preprocessing.ipynb` (`01_Preprocessing.ipynb`).

### Label Encoding

A character vocabulary matching the training labels is defined. Prediction indices from the model classifier are decoded back into text strings using the index-to-character mapping.

In [5]:
# Define the character vocabulary and map indices back to characters for decoding
vocab = [
    '2','3','4','5','6','7','8','9',
    'A','B','C','D','E','F','G','H',
    'J','K','M','N','P','Q','R','S',
    'T','U','V','W','X','Y','Z'
]

idx_to_char = {
    i: ch for i, ch in enumerate(vocab)
}

NUM_CHARS = len(vocab)

### Model Architecture

A ResNet18 architecture with randomly initialized weights was adapted for grayscale sequence recognition. The model incorporates a custom single-channel input layer to process grayscale images and a custom position-aware sequence prediction head of shape `(batch_size, 6, num_characters)`. The network was trained entirely from scratch on the provided dataset without using external data or pretrained weights.

In [ ]:
from torchvision import models
import torch.nn as nn

class ResNetCaptcha(nn.Module):

    def __init__(self, num_chars):

        super().__init__()

        backbone = models.resnet18(weights=None)

        backbone.conv1 = nn.Conv2d(
            1,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        self.features = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,

            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 6))

        self.classifier = nn.Linear(
            512,
            num_chars
        )

    def forward(self, x):

        x = self.features(x)

        x = self.pool(x)

        x = x.squeeze(2)

        x = x.permute(0, 2, 1)

        x = self.classifier(x)

        return x

### Training Strategy

Training is not executed in this notebook. The model parameters were trained from scratch on the provided dataset without transfer learning or pretrained initializations. For testing, the trained checkpoint is loaded and the model is placed in evaluation mode.

### Reproducibility

To guarantee reproducibility, a fixed model weight checkpoint (`final_resnet18_captcha.pth`) is loaded, and the image reading and normalization steps are kept deterministic.

In [7]:
# Load the trained model parameters on CPU and set the model to evaluation mode
device = torch.device("cpu")

model = ResNetCaptcha(NUM_CHARS)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


### Preprocessing

Test images are preprocessed identically to the training images. Each image is read in grayscale, normalized to a float32 representation in the range `[0, 1]`, and converted to a tensor with batch and channel dimensions.

### Error Analysis

Error analysis is performed on the validation split in `01_Preprocessing.ipynb` (`01_Preprocessing.ipynb`). Reviewing misclassified validation images helps identify distortions that the model struggles to predict.

In [8]:
# Load image, normalize pixel values, pass through the model, and decode predicted text
def predict_image(image_path):

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    image = image.astype(np.float32) / 255.0

    image = torch.tensor(
        image,
        dtype=torch.float32
    ).unsqueeze(0).unsqueeze(0)

    with torch.no_grad():

        outputs = model(image)

        preds = outputs.argmax(dim=2)[0]

    text = "".join(
        idx_to_char[idx.item()]
        for idx in preds
    )

    return text

### Dataset & DataLoader Creation

Since the test dataset size is manageable and predictions are made sequentially, images are processed individually without a formal PyTorch DataLoader.

### Test Prediction Generation

Predictions are generated for each test image by taking the argmax of the model output logits across the character dimensions. The decoded text predictions are saved to `submission.csv`.

In [9]:
# Iterate through the test directory, generate sequence predictions, and export to submission.csv
test_images = sorted(
    os.listdir(TEST_FOLDER)
)

predictions = []

for img_name in tqdm(test_images):

    img_path = os.path.join(
        TEST_FOLDER,
        img_name
    )

    pred_text = predict_image(
        img_path
    )

    predictions.append(
        [img_name, pred_text]
    )

submission = pd.DataFrame(
    predictions,
    columns=["image", "prediction"]
)

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(
    "\nSaved:",
    "/kaggle/working/submission.csv"
)

  0%|          | 0/5000 [00:00<?, ?it/s]

           image prediction
0     test-0.png     QVTQ8A
1     test-1.png     7PSW9D
2    test-10.png     7DUP98
3   test-100.png     75Z4WT
4  test-1000.png     QAKZ7V

Saved: /kaggle/working/submission.csv


### Results

The inference pipeline successfully predicts labels for all test images. A sample of the predicted outputs is displayed to verify the sequence format and output correctness.

### Evaluation Metrics

Evaluation metrics such as sequence accuracy and Character Error Rate (CER) are assessed during the validation phase in `01_Preprocessing.ipynb`. No ground truth labels are available for evaluation on this test set.

In [10]:
# Preview the final formatted submission predictions DataFrame
submission.head(10)

,image,prediction
0,test-0.png,QVTQ8A
1,test-1.png,7PSW9D
2,test-10.png,7DUP98
3,test-100.png,75Z4WT
4,test-1000.png,QAKZ7V
5,test-1001.png,R6MERY
6,test-1002.png,CHXX67
7,test-1003.png,9NV2WP
8,test-1004.png,F56TDZ
9,test-1005.png,FFTFRX
